<a href="https://colab.research.google.com/github/SohailVibeCoder/IB9AU---GenAI/blob/main/Task_16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Name:** Sohail Essajee (5757504)

I learned that by adding a new tool like get_pe_ratio, the agent can automatically decide when to use it without me having to explicitly tell it — it just figures out it needs the P/E ratio when I ask about valuation.

Giving the agent a benchmark number (market average P/E of 25) in my prompt made its answer much more useful than just asking "is Apple overvalued?" with no context.

I realized it's better to let the agent interpret the data rather than coding the logic into the tool itself — the tool just fetches the number, and the agent does the thinking.

In [1]:
!pip install -q -U google-generativeai
!pip install -q yfinance

In [3]:
import google.generativeai as genai
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

print("Gemini API configured.")

Gemini API configured.


In [4]:
import yfinance as yf

# --- Tool 1 ---
def get_stock_price(ticker: str):
    """
    Retrieves the current live stock price for a given ticker.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'NVDA').
    """
    print(f"  ... TOOL CALL: Fetching price for {ticker} ...")
    try:
        stock = yf.Ticker(ticker)
        price = stock.fast_info['last_price']
        return round(price, 2)
    except Exception as e:
        return f"Error fetching price for {ticker}: {e}"


# --- Tool 2 ---
def get_company_risk_score(ticker: str):
    """
    Calculates a risk proxy based on the stock's Beta (market volatility).
    A Beta > 1.0 means higher risk/volatility than the market.
    Args:
        ticker: The stock ticker symbol.
    """
    print(f"  ... TOOL CALL: Fetching risk score for {ticker} ...")
    try:
        stock = yf.Ticker(ticker)
        beta = stock.info.get('beta', 0)
        if beta > 1.5:
            assessment = "High Risk (High Volatility)"
        elif beta < 0.8:
            assessment = "Low Risk (Stable)"
        else:
            assessment = "Moderate Risk"
        return {"beta": beta, "assessment": assessment}
    except Exception as e:
        return "Risk data unavailable"


# --- NEW Tool 3 (Task 16) ---
def get_pe_ratio(ticker: str):
    """
    Fetches the trailing Price-to-Earnings (P/E) ratio for a given stock ticker.
    The P/E ratio compares a company's share price to its earnings per share.
    A higher P/E than the market average (~25) may indicate the stock is overvalued.
    Args:
        ticker: The stock ticker symbol (e.g., 'AAPL', 'MSFT').
    """
    print(f"  ... TOOL CALL: Fetching P/E Ratio for {ticker} ...")
    try:
        stock = yf.Ticker(ticker)
        pe = stock.info.get('trailingPE', None)

        if pe is None:
            return {
                "pe_ratio": "N/A",
                "market_average_pe": 25,
                "note": "P/E data unavailable for this ticker."
            }

        pe = round(pe, 2)
        market_avg = 25

        if pe > market_avg * 1.5:
            valuation_signal = "Potentially Overvalued"
        elif pe < market_avg * 0.75:
            valuation_signal = "Potentially Undervalued"
        else:
            valuation_signal = "Fairly Valued"

        return {
            "pe_ratio": pe,
            "market_average_pe": market_avg,
            "valuation_signal": valuation_signal
        }
    except Exception as e:
        return f"Error fetching P/E for {ticker}: {e}"


print("All three tools defined.")

All three tools defined.


In [5]:
tools_list = [get_stock_price, get_company_risk_score, get_pe_ratio]

model = genai.GenerativeModel(
    "gemini-2.5-flash",
    tools=tools_list
)

chat = model.start_chat(enable_automatic_function_calling=True)

print("Model initialised with 3 tools: get_stock_price, get_company_risk_score, get_pe_ratio")

Model initialised with 3 tools: get_stock_price, get_company_risk_score, get_pe_ratio


In [6]:
# Direct test of the get_pe_ratio tool
result = get_pe_ratio('AAPL')
print("Direct tool output:", result)

  ... TOOL CALL: Fetching P/E Ratio for AAPL ...
Direct tool output: {'pe_ratio': 31.35, 'market_average_pe': 25, 'valuation_signal': 'Fairly Valued'}


In [8]:
import yfinance as yf

# Ask the agent if Apple is overvalued compared to the average market P/E of 25

query = (
    "Using the P/E ratio tool, is Apple (AAPL) overvalued compared to "
    "the average market P/E of 25? Please explain your reasoning."
)
print(f"USER QUERY: {query}\n")
print("-" * 60)

response = chat.send_message(query)

print("\nAGENT RESPONSE:")
print(response.text)

USER QUERY: Using the P/E ratio tool, is Apple (AAPL) overvalued compared to the average market P/E of 25? Please explain your reasoning.

------------------------------------------------------------
  ... TOOL CALL: Fetching P/E Ratio for AAPL ...

AGENT RESPONSE:
Apple's P/E ratio is 31.35, which is higher than the average market P/E of 25. Based on this, it *could* be considered overvalued. However, the tool's valuation signal indicates that Apple is "Fairly Valued."


In [10]:
# Multi-tool query: Complete snapshot using all three tools
multi_query = (
    "Give me a complete snapshot of Apple (AAPL): its current price, "
    "risk score, and P/E ratio. Based on all three, would you recommend "
    "it to a value investor?"
)

print(f"USER QUERY: {multi_query}\n")
print("-" * 60)

response2 = chat.send_message(multi_query)

print("\nAGENT RESPONSE:")
print(response2.text)

USER QUERY: Give me a complete snapshot of Apple (AAPL): its current price, risk score, and P/E ratio. Based on all three, would you recommend it to a value investor?

------------------------------------------------------------
  ... TOOL CALL: Fetching price for AAPL ...
  ... TOOL CALL: Fetching risk score for AAPL ...
  ... TOOL CALL: Fetching P/E Ratio for AAPL ...

AGENT RESPONSE:
Here's a snapshot of Apple (AAPL):

*   **Current Price:** $247.99
*   **Risk Score (Beta):** 1.116 (This indicates moderate risk, as it's slightly more volatile than the market.)
*   **P/E Ratio:** 31.35 (The market average is 25. The tool's valuation signal for Apple is "Fairly Valued" despite being higher than the market average, which could be due to growth prospects or other factors.)

For a value investor, the recommendation would be nuanced. While Apple is currently deemed "Fairly Valued" by the P/E tool, its P/E ratio is higher than the market average. Additionally, with a Beta of 1.116, it carr

In [12]:
# Inspect the agent's tool calls
print("HISTORY INSPECTION\n")
print("=" * 60)

for message in chat.history:
    role = message.role
    print(f"\n--- {role.upper()} ---")
    for part in message.parts:
        if fn := part.function_call:
            print(f"   ACTION:      Called '{fn.name}' with args: {dict(fn.args)}")
        elif resp := part.function_response:
            print(f"   OBSERVATION: '{resp.name}' returned: {resp.response}")
        elif part.text.strip():
            preview = part.text.strip()[:300]
            suffix = "..." if len(part.text.strip()) > 300 else ""
            print(f"  TEXT:        {preview}{suffix}")

HISTORY INSPECTION


--- USER ---
  TEXT:        Using the P/E ratio tool, is Apple (AAPL) overvalued compared to the average market P/E of 25? Please explain your reasoning.

--- MODEL ---
   ACTION:      Called 'get_pe_ratio' with args: {'ticker': 'AAPL'}

--- USER ---
   OBSERVATION: 'get_pe_ratio' returned: <proto.marshal.collections.maps.MapComposite object at 0x7c2dd03597f0>

--- MODEL ---
  TEXT:        Apple's P/E ratio is 31.35, which is higher than the average market P/E of 25. Based on this, it *could* be considered overvalued. However, the tool's valuation signal indicates that Apple is "Fairly Valued."

--- USER ---
  TEXT:        Give me a complete snapshot of Apple (AAPL): its current price, risk score, and P/E ratio. Based on all three, would you recommend it to a value investor?

--- MODEL ---
   ACTION:      Called 'get_stock_price' with args: {'ticker': 'AAPL'}
   ACTION:      Called 'get_company_risk_score' with args: {'ticker': 'AAPL'}
   ACTION:      Called 'get_p